In [0]:
from pyspark.sql.types import *

schema = StructType() \
    .add("order_id", IntegerType()) \
    .add("user_id", StringType()) \
    .add("amount", IntegerType())

Step 2: Define Schema

In [0]:
df_bronze = spark.readStream.table("workspace.streaming.ecommerce_bronze")

Step 3: Parse JSON

In [0]:
from pyspark.sql.functions import col, from_json

df_silver = df_bronze.withColumn(
    "parsed",
    from_json(col("value"), schema)
).select(
    col("parsed.*"),
    col("kafka_timestamp"),
    col("ingestion_time")
)

Step 4: Handle bad data

In [0]:
df_silver = df_silver.filter(col("order_id").isNotNull())

Step 5: Write Silver Table

In [0]:
query = df_silver.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/workspace/streaming/checkpoints/ecommerce_pipeline/silver") \
    .trigger(availableNow=True)\
    .table("workspace.streaming.ecommerce_silver")

In [0]:
# %sql
# select * from workspace.streaming.ecommerce_silver order by order_id, ingestion_time desc